# Lecture 28: Designing Experiments — CIs Without the Bootstrap
v.ekc

The payoff of the CLT: confidence intervals straight from a formula, no resampling loop, and the answer to "how many people should I survey?" *before* collecting any data.

**Today's sections:**
1. The CLT method
2. The SD of a 0/1 population

In [ ]:
import matplotlib
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plots
import numpy as np
plots.style.use('fivethirtyeight')

---
## 1. The CLT Method

The CLT says the sample mean is normal with SD = pop SD/√n, and ~95% of a normal curve is within 2 SDs of center. So:

> **95% CI ≈ sample average ± 2 · (pop SD / √n)**, one line of arithmetic. We don't know the pop SD, but the *sample* SD is a good stand-in.

For comparison, Lecture 24's bootstrap gave a 95% confidence interval of (26.90, 27.55) for the average age of the mothers.

In [ ]:
# original sample
births = Table.read_table('baby.csv')
births.show(3)

In [ ]:
sampled_ages = births.column('Maternal Age')
sample_size = len(sampled_ages)
sample_average = np.average(sampled_ages)
sample_SD = np.std(sampled_ages)
sample_size, sample_average, sample_SD

We need to add and subtract 2 * (population SD)/(sample_size ** 0.5) but we don't have the population SD.

In [ ]:
# Try estimating it from the sample
estimated_SD_of_sample_average = sample_SD / (sample_size**0.5)
estimated_SD_of_sample_average

In [ ]:
# Approximate 95% confidence interval for population mean
sample_average - 2*estimated_SD_of_sample_average, sample_average + 2*estimated_SD_of_sample_average

> **Compare:** nearly identical to Lecture 24's bootstrap interval, (26.90, 27.55), for a fraction of the computation. The bootstrap still wins for medians and other statistics the CLT doesn't cover.

### Try It 1: CI for birth weight (optional) 😊

1. Use the CLT method to build an approximate 95% confidence interval for the average **birth weight** in the population.

2. Your interval is only about 2 ounces wide, yet the SD of birth weights is about 18 ounces. How can it be so narrow?

In [ ]:
# 1. CLT interval for Birth Weight


*2. Your interpretation: (double-click to edit)*

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Same four lines with a new column name.*

```python
# 1.
weights = births.column('Birth Weight')
n = len(weights)
w_average = np.average(weights)
w_SD = np.std(weights)
w_average - 2 * w_SD / n ** 0.5, w_average + 2 * w_SD / n ** 0.5    # about (118.4, 120.5)
```

*2. The interval pins down the population **average**, not individual babies: an average of 1,174 weights varies far less than a single weight does, by a factor of √1174.*

</details>

---
## 2. The SD of a 0/1 Population

Proportions are just averages of 0/1 data, and 0/1 populations have a beautiful property: their SD is **at most 0.5**, no matter what.

In [ ]:
# population of size 10

number_of_ones = 2
zero_one_population = np.append(np.ones(number_of_ones), np.zeros(10 - number_of_ones))

print('Standard Deviation:', np.round(np.std(zero_one_population),2))

zero_one_population

Wrap that in a function so we can try every possible mix of ones and zeros:

In [ ]:
def sd_of_zero_one_population(number_of_ones):
    """Returns the SD of a population 
    that has 10 elements: num_ones with value 1 and (10 - num_ones) with value 0"""
    zero_one_population = np.append(np.ones(number_of_ones), 
                                    np.zeros(10 - number_of_ones))
    return np.std(zero_one_population)

In [ ]:
possible_ones = np.arange(11)
zero_one_pop = Table().with_columns(
    'Number of Ones', possible_ones,
    'Proportion of Ones', possible_ones / 10
)
zero_one_pop.show()

Apply the function to every row, then plot SD against proportion:

In [ ]:
sds = zero_one_pop.apply(sd_of_zero_one_population, 'Number of Ones')
zero_one_pop = zero_one_pop.with_column('Pop SD', sds)
zero_one_pop.show()

In [ ]:
zero_one_pop.scatter('Proportion of Ones', 'Pop SD')

Example: a poll of 1,004 randomly sampled voters. Because the SD of a 0/1 population is at most 0.5, the total width of an approximate 95% confidence interval for the proportion of voters who favor a candidate is at most 4 · 0.5/√1004, no matter what that proportion is. Polls usually report the **margin of error**, which is half the width of the interval: about 0.0316, or 3.16 percentage points.

In [ ]:
4 * 0.5 / np.sqrt(1004)

> **The worst case is 0.5** (half ones, half zeros). That means for any proportion, the 95% CI is *at most* `± 2·0.5/√n = ± 1/√n` wide on each side, computable before you collect a single response!

### Try It 2: Design the poll 🗳️

1. You want a 95% confidence interval for a proportion with **total width at most 0.06** (±3 percentage points). What sample size guarantees it, even in the worst case? (*Total width = 4 · 0.5/√n.*)

In [ ]:
# 1. smallest n with 4 * 0.5 / sqrt(n) <= 0.06


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Solve 2/√n ≤ 0.06 → √n ≥ 33.33 → n ≥ 1111.1 → 1112 people. (This is why real polls survey "about 1,100 adults"!)*

```python
# 1.
(2 / 0.06) ** 2          # 1111.1 → need n = 1112
4 * 0.5 / np.sqrt(1112)  # check: ≈ 0.0599 ✓
```

</details>

---
> **Today's takeaway:** CLT turns confidence intervals into arithmetic: mean ± 2·SD/√n, and the 0.5 worst-case SD lets you budget a survey before it exists. This is Homework 8, sections 2 and 3.

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### Sample size for a proportion
```python
# 95% CI total width <= w   =>   n >= (4 * 0.5 / w) ** 2
```